# MULTIPINN: установка и первый запуск

Этот notebook является **канонической инструкцией** по установке и запуску MULTIPINN.

Он согласован с `README.md` и должен использоваться как основная ссылка на туториал в репозитории и во внешних материалах.


## Что зафиксировано в этой инструкции

Чтобы в документации не было расхождений, в этом tutorial используются следующие единые правила:

- **Поддерживаемые версии Python:** `3.8` – `3.11`
- **Рекомендуемая версия для нового окружения:** `3.10`
- **Основная команда установки:** `pip install -e .`
- `make install` рассматривается только как короткая обёртка над той же командой

Ниже команды приводятся для запуска **из корня репозитория**.


## Что такое MULTIPINN

MULTIPINN — это фреймворк для решения задач математической физики с помощью **Physics-Informed Neural Networks (PINNs)**.  
Репозиторий содержит:

- библиотеку `multipinn/` с ядром фреймворка;
- готовые примеры в `examples/`;
- конфигурации на базе **Hydra**;
- средства логирования, сохранения моделей и визуализации результатов.


## Шаг 1. Клонирование репозитория

```bash
git clone https://github.com/multipinn/multipinn.git
cd multipinn
```


## Шаг 2. Создание и активация окружения

### Рекомендуемый вариант: `venv`

**Linux / macOS**
```bash
python3.10 -m venv .venv
source .venv/bin/activate
```

**Windows (PowerShell)**
```powershell
py -3.10 -m venv .venv
.\.venv\Scripts\Activate.ps1
```

Если у вас нет Python 3.10, используйте любую поддерживаемую версию из диапазона `3.8`–`3.11`.


### Альтернатива: `conda`

Если вы используете Conda, создайте окружение так:

```bash
conda create -n multipinn python=3.10
conda activate multipinn
```

Важно: способ создания окружения может отличаться, но **сама установка библиотеки ниже остаётся одинаковой**.


## Шаг 3. Установка MULTIPINN

Основная команда установки:

```bash
python -m pip install --upgrade pip
pip install -e .
```

Эквивалентная короткая команда из `Makefile`:

```bash
make install
```

Если для вашей машины нужен специальный CUDA-сборочный вариант PyTorch, сначала установите подходящий `torch`, а затем выполните `pip install -e .`.


## Шаг 4. Проверка установки

После установки должен успешно работать импорт библиотеки и PyTorch.


In [ ]:
import sys

print("Python:", sys.version.split()[0])

import torch
import multipinn

print("PyTorch:", torch.__version__)
print("MULTIPINN import OK")


## Шаг 5. Быстрый тест

Для быстрой проверки удобно запустить короткий пример с уменьшенным числом эпох.  
Команда ниже запускает `regression_1D` и сохраняет интерактивные `html`-артефакты вместо `png`:

```bash
python -m examples.regression_1D.run_train \
  trainer.num_epochs=5 \
  generator.domain_points=256 \
  visualization.grid_plot_points=256 \
  visualization.save_period=10 \
  visualization.save_mode=html \
  paths.save_dir=./artifacts_smoke/regression_1D
```

Если команда выполняется без ошибок, установка библиотеки и базовый запуск настроены корректно.


## Шаг 6. Первый полноценный пример

## Задача Пуассона


Рассмотрим задачу Пуассона

$$ \nabla^2 u(x, y) = f(x, y),$$

где $\nabla^2 $ - лапласиан $u(x, y)$ , который определяется как:

$$ \nabla^2 u(x, y) = \frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2}$$

Пусть решение уравнения Пуассона задается функцией:

$$ u(x, y) = \cos(2 \pi x) \sin(4 \pi y) + 0.5 x $$

Тогда для конкретного вида решения можно получить следующую тестовую задачу Пуасона:

$$\nabla^2 u(x, y) = -20\pi^2 \sin(4\pi y) \cos(2\pi x) $$

Предположим, что мы ищем решение в области $0 \leq x \leq 0.5$ и $ 0 \leq y \leq 0.5$ и нам заданы граничные условия Дирихле:

1. На границе $ x = 0$:

   $$ u(0, y) = \cos(0) \sin(4 \pi y) + 0.5 \cdot 0 = \sin(4 \pi y). $$

2. На границе $x = 0.5$:

   $$ u(0.5, y) = \cos(\pi) \sin(4 \pi y) + 0.5 \cdot 0.5 = -\sin(4 \pi y) + 0.25. $$

3. На границе $y = 0$:

   $$ u(x, 0) = \cos(2 \pi x) \sin(0) + 0.5 x = 0.5 x. $$

4. На границе $y = 0.5$:

   $$ u(x, 0.5) = \cos(2 \pi x) \sin(2 \pi) + 0.5 x = 0.5 x.$$

Давайте визуализируем решение, чтобы показать нетривиальность выбранной нами задачи. Несмотря на гладкость решения, наличие нескольких локальных максимумов и минимумов может представлять относительную трудность представленному методу.

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Создание сетки значений x и y
x = np.linspace(0, 0.5, 100)
y = np.linspace(0, 0.5, 100)
x, y = np.meshgrid(x, y)

# Вычисление u(x, y)
u = np.cos(2 * np.pi * x) * np.sin(4 * np.pi * y) + 0.5 * x

# Создание 3D поверхности
fig = go.Figure(data=[go.Surface(z=u, x=x, y=y)])

# Обновление макета
fig.update_layout(
    title="3D Surface plot of <i>u(x, y) = cos(2πx)sin(4πy) + 0.5x</i>",
    font_family="Times New Roman",
    font_size=14,
    scene=dict(
        xaxis_title="<i>x</i>", yaxis_title="<i>y</i>", zaxis_title="<i>u(x, y)</i>"
    ),
)

# Отображение графика
fig.show()

In [ ]:
import torch


from multipinn.geometry import (
    RectangleArea,
)  # Помогает задать прямоугольную область, либо отрезок
from multipinn.condition import (
    Condition,
)  # Помогает объединить область с условием задачи
from multipinn.diff import (  # Помогает задать дифференциальное уравнение
    grad,
    unpack,
)

In [ ]:
def Poisson2D1C():

    input_dim = 2  # Количество входов. У нас это x и y
    output_dim = 1  # Количество выходов. Так как считаем одно уравнение, а не систему, то output_dim = 1

    def basic_symbols(arg, model):
        """
        Эта функция принимает аргументы arg и модель model, а затем возвращает ключевые величины для последующего анализа и использования.
        В контексте уравнения Пуассона, она предоставляет:
            - приближенное решение u, вычисленное с использованием модели,
            - координаты x, y, на которых было получено это приближение.
        """
        f = model(arg)
        (u,) = unpack(f)
        x, y = unpack(arg)
        return f, u, x, y

    def inner(arg, model):
        """
        Функция определяет уравнение внутри заданной области.
        В нашем случае - это уравнение Пуассона.

        Метод grad(u, args), который здесь используется, вычисляет производные функции u по каждому из аргументов в args.

        Возвращаемое значение представляет собой набор уравнений, которые необходимо привести к нулю.
        В нашем случае: u_xx + u_yy - f(x,y) = 0, где f(x,y) -- правая часть в уравнении Пуассона.
        """

        f, u, x, y = basic_symbols(arg, model)
        u_x, u_y = unpack(grad(u, arg))
        u_xx, u_xy = unpack(grad(u_x, arg))
        u_yx, u_yy = unpack(grad(u_y, arg))

        eq1 = (
            u_xx
            + u_yy
            + 20
            * torch.pi**2
            * torch.cos(2 * torch.pi * x)
            * torch.sin(4 * torch.pi * y)
        )
        return [eq1]

    """
    Далее идут функции bc{i}, которые сделаны по аналогии с функцией inner, только они задают уравнения, которые должны выполняться на границе
    """

    def bc1(arg, model):
        f, u, x, y = basic_symbols(arg, model)
        return [u - torch.sin(4 * torch.pi * y)]

    def bc2(arg, model):
        f, u, x, y = basic_symbols(arg, model)
        return [u + torch.sin(4 * torch.pi * y) - 0.25]

    def bc3(arg, model):
        f, u, x, y = basic_symbols(arg, model)
        return [u - 0.5 * x]

    def bc4(arg, model):
        f, u, x, y = basic_symbols(arg, model)
        return [u - 0.5 * x]

    # Задаем квадратную область с 0<x<0.5 и 0<y<0.5
    domain = RectangleArea(low=[0, 0], high=[0.5, 0.5])

    # Задаем отрезок x = 0, 0<=y<=0.5
    x_min = RectangleArea(low=[0, 0], high=[0, 0.5])
    # Задаем отрезок x = 0.5, 0<=y<=0.5
    x_max = RectangleArea(low=[0.5, 0], high=[0.5, 0.5])

    # Задаем отрезок y = 0,  0<=x<=0.5
    y_min = RectangleArea(low=[0, 0], high=[0.5, 0])
    # Задаем отрезок y = 0.5, 0<=x<=0.5
    y_max = RectangleArea(low=[0, 0.5], high=[0.5, 0.5])

    pde = [
        # Итоговое уравнение представляет собой набор пар (уравнение, область).
        # Эти пары должны быть инкапсулированы в объекты Condition для корректной работы комплекса MULTIPINN.
        Condition(inner, domain),
        Condition(bc1, x_min),
        Condition(bc2, x_max),
        Condition(bc3, y_min),
        Condition(bc4, y_max),
    ]
    # возвращаем уравнение pde, количество входов input_dim и выходов output_dim
    # входы и выходы также должны быть переданы, чтобы корректно задать слои в нейросети
    return pde, input_dim, output_dim

In [ ]:
import hydra

from multipinn.PINN import PINN
from multipinn.callbacks import (
    curve,
    heatmap,
    save,
    progress,
)

from multipinn.generation.rectangle_generator import *
from multipinn.callbacks.callbacks_organizer import CallbacksOrganizer
from multipinn.neural_network import FNN
from multipinn.trainer import Trainer
from multipinn.utils import set_device

def poisson():

    torch.manual_seed(42)
    np.random.seed(42)

    conditions, input_dim, output_dim = Poisson2D1C()

    set_device()
    model = FNN(layers_all=[input_dim, 128, 128, 128, 128, output_dim])

    generator_domain = UniformGeneratorRect(n_points=20_000, method="uniform")

    generator_bound = UniformGeneratorRect(n_points=2_000, method="uniform")

    generators_config = [{"condition_index": [0], "generator": generator_domain}]

    configure_generators(conditions, generators_config, default_gen=generator_bound)

    pinn = PINN(model=model, conditions=conditions)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, 0.9999)

    grid = heatmap.Grid.from_pinn(pinn, 80_000)

    callbacks = [
        progress.TqdmBar(
            "Epoch {epoch} lr={lr:.2e} Loss={loss_eq} Total={total_loss:.2e}"
        ),
        curve.LossCurve("/path/to/loss/", 100),
        save.SaveModel("/path/to/model/", period=100),
        heatmap.HeatmapPrediction(
            grid=grid, period=100, save_dir="/path/to/Heatmap/", save_mode="png"
        ),
    ]

    trainer = Trainer(
        num_epochs=1000,
        update_grid_every=100,
        pinn=pinn,
        optimizer=optimizer,
        scheduler=scheduler,
        callbacks_organizer=CallbacksOrganizer(callbacks),
    )

    trainer.train()

poisson()

Так же можно запустить задачу Пуассона и кодовым скриптом так:

```bash
python -m examples.poisson_2D_1C.run_train
```

Для более короткого локального прогона можно временно переопределить параметры Hydra:

```bash
python -m examples.poisson_2D_1C.run_train \
  trainer.num_epochs=100 \
  generator.domain_points=2000 \
  generator.bound_points=400 \
  visualization.grid_plot_points=1000 \
  visualization.save_period=100 \
  visualization.save_mode=html
```


## Где настраивать задачу

У большинства примеров структура одинаковая:

```text
examples/<example_name>/
├── configs/config.yaml
├── problem.py
└── run_train.py
```

Назначение файлов:

- `configs/config.yaml` — гиперпараметры, генераторы точек, пути сохранения, визуализация;
- `problem.py` — уравнения, геометрия, граничные и начальные условия;
- `run_train.py` — точка входа, которая создаёт модель, генераторы, callbacks и запускает обучение.


## Какие параметры меняют чаще всего

В `config.yaml` обычно настраиваются:

- `problem` — параметры физической постановки;
- `model` — архитектура сети;
- `regularization` — способ балансировки потерь;
- `generator` — число внутренних и граничных точек;
- `trainer` — число эпох, seed, частота обновления;
- `visualization` — плотность сетки, период сохранения, формат артефактов;
- `optimizer` и `scheduler` — оптимизатор и шаг обучения;
- `paths` — директория для результатов.

Это позволяет изменять большинство параметров без правки исходного кода.


## Где искать результаты

Результаты сохраняются в каталог, заданный в `paths.save_dir`.

Обычно там появляются:

- `used_config.yaml` — копия реально использованной конфигурации;
- веса модели (`.pth`);
- кривые обучения;
- визуализации (`html` и/или `png`);
- дополнительные файлы, создаваемые callbacks.

Если вам не нужны статические изображения, удобнее использовать:

```yaml
visualization:
  save_mode: html
```

Это снимает часть требований к локальной среде для PNG-экспорта.


## Важная заметка про GNN и mesh-примеры

Базовая установка тянет `torch_geometric` через `requirements.txt`.  
При этом часть graph- и mesh-сценариев может дополнительно требовать бинарные пакеты PyG, совместимые с вашей версией **PyTorch** и **CUDA**:

- `pyg_lib`
- `torch_scatter`
- `torch_sparse`
- `torch_cluster`
- `torch_spline_conv`

Если вы видите ошибки, связанные с `torch_scatter`, `torch_cluster`, `knn_graph` или `radius_graph`, проверьте версию Torch/CUDA:

```bash
python -c "import torch; print(torch.__version__, torch.version.cuda)"
```

После этого установите совместимые пакеты PyG и повторите запуск.


## Установка для разработки

Если вы не только запускаете примеры, но и дорабатываете библиотеку, используйте dev-установку:

```bash
pip install -e ".[dev]"
```

Эквивалентная короткая команда:

```bash
make install.dev
```

Полезные команды:

```bash
make install-pre-commit
make test
```
